# 실습 1 | Minimal Agent Loop

**학습 목표**
- 모델이 현재 정보에 따라 Tool과 입력값을 선택하는 과정을 확인한다.
- Python 함수가 실행한 Tool Result를 다음 모델 호출에 전달한다.
- 추가 Tool Call이 없을 때 종료하는 Agent Loop를 직접 실행하고 관찰한다.

이번 실습은 완성된 코드를 위에서 아래로 실행합니다. 필요한 구성 요소는 이 Notebook과 `tools/`에 있으며, 다른 실습 파일을 실행할 필요가 없습니다.


---
## 1. 실행 환경 확인

Python 3.12 가상환경에서 저장소 루트의 `requirements.txt`를 설치하고, `.env.example`을 참고해 루트에 `.env`를 준비합니다. 
<br>`OPENAI_API_KEY`가 필요하고, 한국어 위키백과 검색에는 인터넷 연결이 필요합니다. Notebook 커널도 같은 가상환경을 선택하세요.

다음 셀은 저장소 위치를 찾아 공통 Tool을 불러옵니다. API 키 값은 출력하지 않습니다.


In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

PROJECT_ROOT = next(
    (
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "requirements.txt").is_file() and (path / "tools").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("저장소 루트 또는 workshop/ 폴더에서 Notebook을 실행하세요.")

sys.path.insert(0, str(PROJECT_ROOT))
from tools.read_file import read_file
from tools.wikipedia_search import wikipedia_search

load_dotenv(PROJECT_ROOT / ".env")
if not os.getenv("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY") == "your-openai-api-key":
    raise RuntimeError("저장소 루트의 .env에 OPENAI_API_KEY를 설정하세요.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
client = OpenAI()
print(f"Python: {sys.version_info.major}.{sys.version_info.minor}")
print(f"프로젝트: {PROJECT_ROOT}")
print(f"모델: {MODEL}")


> **결과 분석:** 프로젝트 경로와 모델 이름이 보이면 준비가 끝났습니다. Python 버전이 3.12가 아니라면 선택한 Notebook 커널을 확인하세요.


---
## 2. Tool을 모델과 Python 함수에 연결하기

모델에게는 Tool의 이름·설명·입력 형식(schema)을 알려줍니다. 
<br>실제 함수 실행은 `TOOL_REGISTRY`가 담당합니다. 모델이 `read_file`을 요청한다고 해서 API가 자동으로 로컬 파일을 읽지는 않습니다.

`read_file`은 `workspace/sample.txt`를 읽고, `wikipedia_search`는 한국어 위키백과의 검색 결과와 URL을 돌려줍니다. 아래 출력에서 두 Tool 이름이 schema와 registry에 모두 있는지 확인하세요.


In [ ]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "name": "read_file",
        "description": "저장소 안의 UTF-8 텍스트 파일을 읽습니다. 기사 내용을 확인할 때 사용합니다.",
        "parameters": {
            "type": "object",
            "properties": {"path": {"type": "string", "description": "저장소 루트 기준 상대 경로"}},
            "required": ["path"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "wikipedia_search",
        "description": "한국어 위키백과에서 인물을 검색해 소개와 문서 URL을 가져옵니다.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "기사에서 확인한 선수 이름"}},
            "required": ["query"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]
TOOL_REGISTRY = {
    "read_file": read_file,
    "wikipedia_search": wikipedia_search,
}
print("모델에 제공하는 Tool:", [tool["name"] for tool in TOOL_SCHEMAS])
print("Python에서 실행할 함수:", list(TOOL_REGISTRY))


> **결과 분석:** 두 출력 목록에 `read_file`과 `wikipedia_search`가 동일하게 나타납니다. 모델에 제공한 Tool 이름과 Python에서 실행할 함수 이름이 일치함을 확인할 수 있습니다.


---
## 3. 첫 Model Call에서 Tool 선택 관찰하기

사용자 요청만 모델에 전달합니다. 기사 본문은 전달하지 않으므로 모델은 필요하다면 `read_file`을 요청해야 합니다. 
<br>모델이 고른 Tool 이름과 인자를 확인하세요. Tool 호출 순서를 Python 코드에서 지정하지 않습니다.

> **실행 참고:** 모델의 답변과 Tool 호출 대상·순서는 실행마다 달라질 수 있습니다.


In [ ]:
TASK = (
    "workspace/sample.txt의 기사를 읽고, 등장하는 야구선수 한 명을 한국어 위키백과에서 "
    "검색해 기사 속 활약과 어떤 선수인지 함께 설명해줘. 기사와 위키백과 문서 링크도 제시해줘."
)
INSTRUCTIONS = (
    "기사 본문과 한국어 위키백과 검색 결과를 실제 Tool로 확인한 후 답하세요. "
    "기사에서 활약이 분명한 선수 한 명만 고르고, 그 이름으로 wikipedia_search를 한 번만 요청하세요. "
    "확인되지 않은 선수 정보는 추측하지 말고, 검색 결과가 없으면 그 사실을 밝히세요."
)

def ask_model(input_items):
    return client.responses.create(
        model=MODEL,
        instructions=INSTRUCTIONS,
        input=input_items,
        tools=TOOL_SCHEMAS,
        tool_choice="auto",
    )

def show_model_result(response):
    calls = [item for item in response.output if item.type == "function_call"]
    print("[MODEL]")
    for call in calls:
        print(f"[TOOL CALL] {call.name}({call.arguments})")
    if not calls:
        print("[FINAL ANSWER]")
        print(response.output_text)
    return calls

conversation = [{"role": "user", "content": TASK}]
print("[USER]", TASK)
first_response = ask_model(conversation)
first_calls = show_model_result(first_response)


> **결과 분석:** `[USER]`는 전달한 요청이고, `[MODEL]` 뒤의 `[TOOL CALL]`은 모델이 요청한 Tool 이름과 JSON 인자입니다. 이 셀에서는 호출 요청만 표시하며 Tool 결과는 아직 출력되지 않습니다.


---
## 4. Tool을 실행하고 Result를 Context에 넣기

모델이 요청한 이름을 registry에서 찾고 JSON 인자를 Python 함수에 전달합니다. 
<br>반환값은 같은 `call_id`를 가진 `function_call_output`으로 대화 목록에 추가합니다. 다음 셀에서 Tool Result 내용과 대화 목록 끝의 항목 형식을 확인하세요.


In [ ]:
def execute_tool_calls(calls):
    outputs = []
    for call in calls:
        arguments = json.loads(call.arguments)
        result = TOOL_REGISTRY[call.name](**arguments)
        output = {
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": result,
        }
        outputs.append(output)
        print(f"[TOOL RESULT] {call.name}")
        print(result)
    return outputs

if first_calls:
    conversation.extend(first_response.output)
    first_outputs = execute_tool_calls(first_calls)
    conversation.extend(first_outputs)
    print("다음 Model Call에 전달할 마지막 항목의 type:", conversation[-1]["type"])
else:
    print("모델이 Tool을 요청하지 않았습니다. 요청과 지시문을 확인한 뒤 다시 실행하세요.")


> **결과 분석:** `[TOOL RESULT]` 아래에는 실행한 함수의 반환값이 출력됩니다. 마지막 항목의 `type`이 `function_call_output`이면 Tool 반환값이 다음 모델 입력에 포함된 것을 확인할 수 있습니다.


---
## 5. Tool Result를 받은 모델의 다음 결정 확인하기

업데이트된 대화 목록으로 모델을 다시 호출합니다. 
<br>추가 Tool Call이 생기면 그 이름과 인자를 확인하세요. 추가 호출이 없다면 모델의 최종 답변이 표시됩니다.


In [ ]:
if first_calls:
    second_response = ask_model(conversation)
    second_calls = show_model_result(second_response)


> **고찰:** Tool 결과를 받은 모델도 추가 Tool을 요청할 수 있으므로, 모델을 두 번 호출하는 것만으로 과제가 끝난다고 가정할 수 없습니다. 필요한 호출을 처리하고 결과를 다시 전달하는 반복 구조가 필요합니다.


---
## 6. 완성된 Agent Loop 실행하기

앞에서 확인한 세 동작, 즉 **모델 호출 → Tool 실행 → Result 전달**을 반복합니다. 
<br>각 반복에서 모델이 요청한 Tool을 실행하고, 더 이상 Tool Call이 없으면 최종 답변을 반환합니다. 이 셀은 새로운 요청으로 시작하여 모델 호출·Tool 실행·결과 전달 과정을 처음부터 다시 수행합니다.


In [ ]:
def run_agent(task):
    input_items = [{"role": "user", "content": task}]
    print("[USER]", task)
    while True:
        response = ask_model(input_items)
        calls = show_model_result(response)
        if not calls:
            return response.output_text

        input_items.extend(response.output)
        input_items.extend(execute_tool_calls(calls))

final_answer = run_agent(TASK)


> **결과 분석:** `[MODEL]`, `[TOOL CALL]`, `[TOOL RESULT]`의 출력 순서에서 모델과 Tool의 상호작용을 확인할 수 있습니다. `[FINAL ANSWER]`는 추가 Tool Call 없이 반환한 최종 답변입니다.


---
## 전체 정리

일반적인 한 번의 LLM 호출과 달리, 이번 Agent는 모델의 Tool 요청을 Python 함수로 실행하고 그 결과를 다시 모델에 전달합니다. 실습 2에서는 이 실행에 중단 조건, 오류 처리와 기록을 더해 Harness의 역할을 살펴봅니다.

참고: [OpenAI Function Calling 공식 문서](https://developers.openai.com/api/docs/guides/function-calling), [MediaWiki Action API](https://www.mediawiki.org/wiki/API:Main_page)
